# Imports

In [66]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import optuna
import shap

import umap
import hdbscan

from joblib import dump, load
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, recall_score

# Constants

In [67]:
RANDOM_SEED = 42

# Inputs

In [ ]:
DB_PATH = "../data/sqlite/data.db"
TABLE   = "network_data"

# Samples per class (defined by the `label` column).
# - int  → same cap for every class
# - None → all available data (careful: may crash on large datasets)
SAMPLES_PER_CLASS: int | None = 10_000
# Per-class overrides — take precedence over SAMPLES_PER_CLASS.
SAMPLES_PER_CLASS_OVERRIDE: dict = {}  # e.g. {"benign": 20_000}

# Binary sampling: (n_benign, n_attack) — overrides SAMPLES_PER_CLASS when set.
# Loads n_benign rows from BENIGN_LABEL and n_attack rows from all other classes combined.
# Example: (50_000, 50_000) → 50k benign + 50k attack (pooled from all attack classes)
BINARY_SAMPLING: "tuple[int, int] | None" = None
BINARY_SAMPLING = [20000, 20000]

BENIGN_LABEL: str = "benign"

# Data

## Load Data

In [69]:
def load_dataset_from_db(
    db_path: str,
    table: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    binary_sampling: "tuple[int, int] | None" = None,
    benign_label: str = "benign",
) -> pd.DataFrame:
    """
    Load data from SQLite, sampling per class directly in the DB so that
    only the selected rows are pulled into memory.

    SQLite handles sampling with ORDER BY RANDOM() LIMIT n, avoiding the
    need to load the full dataset into pandas before filtering.

    When binary_sampling=(n_benign, n_attack) is provided, samples n_benign
    rows from benign_label and n_attack rows from all other classes pooled
    together, ignoring samples_per_class and override.
    """
    conn = sqlite3.connect(db_path)

    if binary_sampling is not None:
        n_benign, n_attack = binary_sampling

        q_benign = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
        df_benign = pd.read_sql(q_benign, conn, params=(benign_label, n_benign))
        print(f"  {benign_label}: {len(df_benign):,} rows")

        q_attack = f'SELECT * FROM "{table}" WHERE label != ? ORDER BY RANDOM() LIMIT ?'
        df_attack = pd.read_sql(q_attack, conn, params=(benign_label, n_attack))
        attack_counts = df_attack["label"].value_counts().to_dict()
        for cls, cnt in sorted(attack_counts.items()):
            print(f"  {cls}: {cnt:,} rows")

        conn.close()
        result = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=42)
        print(f"\nTotal: {len(result):,} rows")
        return result

    classes = pd.read_sql(
        f'SELECT DISTINCT label FROM "{table}" WHERE label IS NOT NULL', conn
    )["label"].tolist()
    print(f"Classes: {sorted(classes)}")

    frames = []
    for cls in classes:
        n = override.get(cls, samples_per_class)

        if n is None:
            query = f'SELECT * FROM "{table}" WHERE label = ?'
            df_cls = pd.read_sql(query, conn, params=(cls,))
        else:
            query = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(query, conn, params=(cls, n))

        frames.append(df_cls)
        print(f"  {cls}: {len(df_cls):,} rows")

    conn.close()

    result = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42)
    print(f"\nTotal: {len(result):,} rows")
    return result

In [70]:
def load_dataset_from_csv(
    data_root: str,
    dataset: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Alternative: load from CSV files directly.
    Only use for small datasets — large ones will exhaust RAM.
    """
    import glob
    dataset_path = os.path.join(data_root, dataset)
    csv_paths = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found in: {dataset_path}")

    df = pd.concat((pd.read_csv(p) for p in csv_paths), ignore_index=True)

    if samples_per_class is None and not override:
        return df

    sampled = []
    for cls in df["label"].unique():
        n = override.get(cls, samples_per_class)
        cls_df = df[df["label"] == cls]
        sampled.append(cls_df.sample(min(n, len(cls_df)), random_state=random_state) if n else cls_df)

    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

In [71]:
df = load_dataset_from_db(
    DB_PATH,
    TABLE,
    samples_per_class=SAMPLES_PER_CLASS,
    override=SAMPLES_PER_CLASS_OVERRIDE,
    binary_sampling=BINARY_SAMPLING,
    benign_label=BENIGN_LABEL,
)
df.head()

  benign: 20,000 rows
  bruteforce: 1 rows
  ddos: 14,237 rows
  dos: 5,753 rows
  malware: 6 rows
  web: 3 rows

Total: 40,000 rows


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,1.737650e+09,49.125257,13.027922,0.325698,0.325698,...,23.599983,23.599983,23.599983,0.0,6.427591,6.427591,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,1.757453e+09,0.004707,16996.470469,424.911762,212.455881,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,1.737816e+09,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,1.757456e+09,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,1.737400e+09,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,dos


## Data Cleaning

### Removing useless features

Some columns were discarded because they did not provide useful information for training, or because they biased the information:
- Timestamp

In [72]:
removed_features: dict[str, list[str]] = {}

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

USELESS_FEATURES = ["timestamp"]#, "src_ip", "dst_ip"]
actually_dropped = [c for c in USELESS_FEATURES if c in df.columns]
df = df.drop(columns=actually_dropped)
removed_features["useless_features"] = actually_dropped

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

Before:
  Shape: (40000, 83)
  Columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_seg

### Removing spaces from feature names

In [73]:
spaces_before = [c for c in df.columns if c != c.strip()]
print("Before - columns with spaces:", spaces_before)

df.columns = df.columns.str.strip()

spaces_after = [c for c in df.columns if c != c.strip()]
print("After  - columns with spaces:", spaces_after)
df.head()

Before - columns with spaces: []
After  - columns with spaces: []


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,49.125257,13.027922,0.325698,0.325698,0.000000,...,23.599983,23.599983,23.599983,0.0,6.427591,6.427591,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,0.004707,16996.470469,424.911762,212.455881,212.455881,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,dos


### Removing rows with inoperative values

In [74]:
numeric_df = df.select_dtypes(include="number")

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")

df = df.replace([np.inf, -np.inf], np.nan).dropna()

numeric_df = df.select_dtypes(include="number")
print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")
df.head()

Before:
  Shape: (40000, 82)
  NaN count:  0
  Inf count:  0

After:
  Shape: (40000, 82)
  NaN count:  0
  Inf count:  0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,49.125257,13.027922,0.325698,0.325698,0.000000,...,23.599983,23.599983,23.599983,0.0,6.427591,6.427591,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,0.004707,16996.470469,424.911762,212.455881,212.455881,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,dos


### Removing duplicate rows

In [75]:
print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

df = df.drop_duplicates()

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")
df.head()

Before:
  Shape: (40000, 82)
  Duplicate rows: 643

After:
  Shape: (39357, 82)
  Duplicate rows: 0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,49.125257,13.027922,0.325698,0.325698,0.000000,...,23.599983,23.599983,23.599983,0.0,6.427591,6.427591,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,0.004707,16996.470469,424.911762,212.455881,212.455881,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0,dos


### Removing highly correlated features

In [76]:
CORR_THRESHOLD = 0.95

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f"  Dropped (corr > {CORR_THRESHOLD}): {to_drop}")

df = df.drop(columns=to_drop)
removed_features["high_correlation"] = to_drop

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (39357, 82)
  Dropped (corr > 0.95): ['flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_bwd_pkts', 'bwd_pkt_len_std', 'pkt_len_min', 'pkt_len_mean', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'fwd_iat_tot', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_psh_flags', 'bwd_psh_flags', 'psh_flag_cnt', 'ack_flag_cnt', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'active_mean', 'active_std', 'idle_max', 'idle_mean']

After:
  Shape: (39357, 59)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,tot_fwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,...,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_min,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,49.125257,13.027922,16,640,0,...,0,0,512,-1,23.599983,23.599983,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,0.004707,16996.470469,1,40,40,...,1,40,62,4350,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,0.000000,0.000000,1,40,0,...,0,0,512,-1,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,0.000000,0.000000,1,71,0,...,0,0,-1,-1,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,0.000000,0.000000,1,40,0,...,0,0,512,-1,0.000000,0.000000,0.000000,0.0,0,dos


### Removing near-zero variance features

In [77]:
VARIANCE_THRESHOLD = 1e-4

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

variances = df[numeric_cols].var()
low_var_cols = variances[variances < VARIANCE_THRESHOLD].index.tolist()
print(f"  Dropped (var < {VARIANCE_THRESHOLD}): {low_var_cols}")

df = df.drop(columns=low_var_cols)
removed_features["near_zero_variance"] = low_var_cols

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (39357, 59)
  Dropped (var < 0.0001): ['fwd_urg_flags', 'bwd_urg_flags', 'urg_flag_cnt']

After:
  Shape: (39357, 56)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,tot_fwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,...,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_min,idle_std,cwr_flag_count,label
32823,192.168.1.104,192.168.1.52,21140,80,6,49.125257,13.027922,16,640,0,...,0,0,512,-1,23.599983,23.599983,6.427591,0.0,0,ddos
16298,47.88.5.14,192.168.1.50,443,40916,6,0.004707,16996.470469,1,40,40,...,1,40,62,4350,0.000000,0.000000,0.000000,0.0,0,benign
28505,192.168.1.104,192.168.1.16,36737,1883,6,0.000000,0.000000,1,40,0,...,0,0,512,-1,0.000000,0.000000,0.000000,0.0,0,ddos
6689,192.168.1.1,224.0.0.251,39854,5353,17,0.000000,0.000000,1,71,0,...,0,0,-1,-1,0.000000,0.000000,0.000000,0.0,0,benign
26893,192.168.1.100,192.168.1.13,10767,1883,6,0.000000,0.000000,1,40,0,...,0,0,512,-1,0.000000,0.000000,0.000000,0.0,0,dos


In [78]:
final_features = df.columns.tolist()
all_removed    = [col for cols in removed_features.values() for col in cols]

sep  = "=" * 60
sep2 = "-" * 60
lines = [
    sep,
    "FEATURE REPORT — DATA CLEANING SUMMARY",
    sep,
    "",
    f"Final features used ({len(final_features)}):",
]
for feat in sorted(final_features):
    lines.append(f"  + {feat}")

lines += ["", f"Excluded features ({len(all_removed)}):"]
for reason, cols in removed_features.items():
    if cols:
        lines += ["", f"  [{reason}]"]
        for col in sorted(cols):
            lines.append(f"    - {col}")

report = "\n".join(lines) + "\n"
print(report)

os.makedirs("../docs", exist_ok=True)
with open("../docs/features_report.txt", "w") as fh:
    fh.write(report)

print("Saved → ../docs/features_report.txt")

FEATURE REPORT — DATA CLEANING SUMMARY

Final features used (56):
  + active_max
  + active_min
  + bwd_blk_rate_avg
  + bwd_byts_b_avg
  + bwd_header_len
  + bwd_iat_max
  + bwd_iat_mean
  + bwd_iat_min
  + bwd_iat_std
  + bwd_iat_tot
  + bwd_pkt_len_max
  + bwd_pkt_len_mean
  + bwd_pkt_len_min
  + bwd_pkts_b_avg
  + cwr_flag_count
  + down_up_ratio
  + dst_ip
  + dst_port
  + ece_flag_cnt
  + fin_flag_cnt
  + flow_byts_s
  + flow_duration
  + flow_iat_max
  + flow_iat_mean
  + flow_iat_min
  + flow_iat_std
  + fwd_blk_rate_avg
  + fwd_byts_b_avg
  + fwd_header_len
  + fwd_iat_max
  + fwd_iat_std
  + fwd_pkt_len_max
  + fwd_pkt_len_mean
  + fwd_pkt_len_min
  + fwd_pkt_len_std
  + fwd_pkts_b_avg
  + idle_min
  + idle_std
  + init_bwd_win_byts
  + init_fwd_win_byts
  + label
  + pkt_len_max
  + pkt_len_std
  + pkt_len_var
  + protocol
  + rst_flag_cnt
  + src_ip
  + src_port
  + subflow_bwd_byts
  + subflow_bwd_pkts
  + subflow_fwd_byts
  + subflow_fwd_pkts
  + syn_flag_cnt
  + tot_fwd_

# Phase 1 — Binary Classification (Benign vs Malicious)

In [79]:
y = df["label"].apply(lambda x: "benign" if x == "benign" else "attack")
X = df.drop(columns=df.select_dtypes(exclude="number").columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

A prioridade aqui é o **recall**, para minimizar falsos negativos — é preferível suspeitar de tráfego benigno do que deixar um ataque passar como benigno.

In [80]:
def objective(trial):
    threshold = trial.suggest_float("threshold", 0.01, 0.5)

    params = {
        "objective": "multiclass",
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 128, 512),
        "class_weight": "balanced",
        "verbosity": -1,
        "force_row_wise": True,
        "n_jobs": 3,
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)

        attack_idx = list(model.classes_).index("attack")
        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        scores.append(recall_score(y_val_bin, y_pred))

    return np.mean(scores)

In [81]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-05-02 14:22:32,018] A new study created in memory with name: no-name-0bd98fe6-9f5b-42b5-a4da-a8ee9c5a81b5
Best trial: 0. Best value: 0.999061:   5%|▌         | 1/20 [00:00<00:16,  1.15it/s, 0.87/1800 seconds]

[I 2026-05-02 14:22:32,887] Trial 0 finished with value: 0.9990613853951568 and parameters: {'threshold': 0.2263162613342965, 'num_leaves': 149, 'max_depth': 0, 'min_child_samples': 58, 'min_split_gain': 0.6017286050387032, 'learning_rate': 0.08711038204274847, 'n_estimators': 365, 'subsample': 0.78371776786241, 'subsample_freq': 1, 'colsample_bytree': 0.9192255176763754, 'reg_alpha': 1.5262771241992927, 'reg_lambda': 7.843660630864536e-05, 'max_bin': 352}. Best is trial 0 with value: 0.9990613853951568.


Best trial: 0. Best value: 0.999061:  10%|█         | 2/20 [00:01<00:15,  1.20it/s, 1.68/1800 seconds]

[I 2026-05-02 14:22:33,700] Trial 1 finished with value: 0.9990613853951568 and parameters: {'threshold': 0.20598920803000637, 'num_leaves': 148, 'max_depth': 16, 'min_child_samples': 97, 'min_split_gain': 0.6506852118403805, 'learning_rate': 0.05250275660521531, 'n_estimators': 229, 'subsample': 0.6765794369944613, 'subsample_freq': 7, 'colsample_bytree': 0.6246249847634979, 'reg_alpha': 0.04093618014658798, 'reg_lambda': 8.635187314934765, 'max_bin': 475}. Best is trial 0 with value: 0.9990613853951568.


Best trial: 0. Best value: 0.999061:  15%|█▌        | 3/20 [00:04<00:27,  1.60s/it, 4.18/1800 seconds]

...

[I 2026-05-02 14:23:07,983] Trial 17 finished with value: 0.9991865340091359 and parameters: {'threshold': 0.15399557125184374, 'num_leaves': 114, 'max_depth': 7, 'min_child_samples': 40, 'min_split_gain': 0.5909167184047803, 'learning_rate': 0.01580484115598701, 'n_estimators': 391, 'subsample': 0.7512883442529024, 'subsample_freq': 1, 'colsample_bytree': 0.7742516729114778, 'reg_alpha': 5.132019526224656e-07, 'reg_lambda': 6.903259615728645e-07, 'max_bin': 238}. Best is trial 12 with value: 0.9998122770790313.


Best trial: 12. Best value: 0.999812:  95%|█████████▌| 19/20 [00:39<00:02,  2.62s/it, 39.30/1800 seconds]

[I 2026-05-02 14:23:11,322] Trial 18 finished with value: 0.9985607909392403 and parameters: {'threshold': 0.35420359235927745, 'num_leaves': 57, 'max_depth': 10, 'min_child_samples': 18, 'min_split_gain': 0.7338012732602717, 'learning_rate': 0.0075421623053136926, 'n_estimators': 608, 'subsample': 0.8148268316544983, 'subsample_freq': 3, 'colsample_bytree': 0.8587049735490497, 'reg_alpha': 4.074513652228157e-05, 'reg_lambda': 1.0072120103580965e-05, 'max_bin': 313}. Best is trial 12 with value: 0.9998122770790313.


Best trial: 12. Best value: 0.999812: 100%|██████████| 20/20 [00:40<00:00,  2.02s/it, 40.35/1800 seconds]

[I 2026-05-02 14:23:12,372] Trial 19 finished with value: 0.9996245541580627 and parameters: {'threshold': 0.05651905658848411, 'num_leaves': 125, 'max_depth': 2, 'min_child_samples': 46, 'min_split_gain': 0.8630011580746806, 'learning_rate': 0.014136491096930583, 'n_estimators': 294, 'subsample': 0.9992801975543345, 'subsample_freq': 5, 'colsample_bytree': 0.6812889968645943, 'reg_alpha': 0.0024697225918859536, 'reg_lambda': 1.2555515814479613e-07, 'max_bin': 195}. Best is trial 12 with value: 0.9998122770790313.


In [82]:
print("Best recall:", study.best_value)
print("Best params:", study.best_params)

Best recall: 0.9998122770790313
Best params: {'threshold': 0.019374508892715848, 'num_leaves': 96, 'max_depth': 4, 'min_child_samples': 33, 'min_split_gain': 0.8056427497850192, 'learning_rate': 0.0060759871639781225, 'n_estimators': 570, 'subsample': 0.8650076354457568, 'subsample_freq': 2, 'colsample_bytree': 0.8364734417819358, 'reg_alpha': 9.36252219062257e-05, 'reg_lambda': 5.024553692642752e-07, 'max_bin': 274}


In [83]:
best = study.best_params.copy()
threshold = best.pop("threshold")

model = lgb.LGBMClassifier(**best)

In [84]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,96
,max_depth,4
,learning_rate,0.0060759871639781225
,n_estimators,570
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.8056427497850192
,min_child_weight,0.001
,min_child_samples,33


In [85]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      attack       1.00      1.00      1.00      3996
      benign       1.00      1.00      1.00      3876

    accuracy                           1.00      7872
   macro avg       1.00      1.00      1.00      7872
weighted avg       1.00      1.00      1.00      7872



In [86]:
recall_score(y_test, y_pred, pos_label="attack")

0.997997997997998

In [87]:
dump(model, "../models/binary_classifier.pkl")

['../models/binary_classifier.pkl']

# Phase 2 — Multiclassification

# Phase 3 — Clustering